# Causal Critic Training on CRD3 Dataset

This notebook trains a lightweight DeBERTa-v3-small model for evaluating causal consistency between player actions and DM responses in D&D narratives.

## Overview
- **Model**: DeBERTa-v3-small (fine-tuned for NLI-based causal consistency)
- **Dataset**: CRD3 NPC Dialogues with context
- **Task**: Natural Language Inference (entailment scoring)
- **Goal**: Evaluate if DM responses logically follow from player actions

## Methodology
1. Load CRD3 dialogue data
2. Create positive pairs (context → response)
3. Generate negative pairs (mismatched context-response)
4. Fine-tune DeBERTa-v3-small on NLI task
5. Evaluate model performance with comprehensive metrics

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install -q transformers datasets torch scikit-learn matplotlib seaborn pandas numpy tqdm

In [ ]:
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
from collections import Counter

import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
from datasets import Dataset, DatasetDict
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

## 2. Load and Explore CRD3 Dataset

In [ ]:
# Load CRD3 NPC dialogues
# In Kaggle: /kaggle/input/crd3-npc-dialogues/crd3_npc_dialogues.json
# Local: crd3_npc_dialogues.json

DATA_PATH = "/kaggle/input/crd3-npc-dialogues/crd3_npc_dialogues.json"  # Kaggle path
# DATA_PATH = "crd3_npc_dialogues.json"  # Uncomment for local testing

print("Loading CRD3 dataset...")
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    crd3_data = json.load(f)

print(f"\n✓ Loaded {len(crd3_data):,} dialogue entries")
print("\nSample entry:")
print(json.dumps(crd3_data[0], indent=2))

In [ ]:
# Exploratory Data Analysis
df = pd.DataFrame(crd3_data)

print("="*60)
print("DATASET STATISTICS")
print("="*60)
print(f"\nTotal entries: {len(df):,}")
print(f"Unique characters: {df['character'].nunique()}")
print(f"Unique episodes: {df['episode'].nunique()}")
print(f"\nText length statistics:")
df['text_length'] = df['text'].str.len()
df['context_length'] = df['context'].str.len()
print(df[['text_length', 'context_length']].describe())

print("\nTop 10 characters by dialogue count:")
print(df['character'].value_counts().head(10))

In [ ]:
# Visualize data distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Character distribution
top_chars = df['character'].value_counts().head(15)
axes[0, 0].barh(range(len(top_chars)), top_chars.values)
axes[0, 0].set_yticks(range(len(top_chars)))
axes[0, 0].set_yticklabels(top_chars.index)
axes[0, 0].set_xlabel('Number of Dialogues')
axes[0, 0].set_title('Top 15 Characters by Dialogue Count')
axes[0, 0].invert_yaxis()

# Text length distribution
axes[0, 1].hist(df['text_length'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Text Length (characters)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Distribution of Response Text Length')
axes[0, 1].axvline(df['text_length'].median(), color='red', linestyle='--', label='Median')
axes[0, 1].legend()

# Context length distribution
axes[1, 0].hist(df['context_length'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1, 0].set_xlabel('Context Length (characters)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Context Length')
axes[1, 0].axvline(df['context_length'].median(), color='red', linestyle='--', label='Median')
axes[1, 0].legend()

# Episode distribution
episode_counts = df['episode'].value_counts().head(20)
axes[1, 1].bar(range(len(episode_counts)), episode_counts.values)
axes[1, 1].set_xlabel('Episode')
axes[1, 1].set_ylabel('Number of Dialogues')
axes[1, 1].set_title('Top 20 Episodes by Dialogue Count')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\n✓ Data exploration complete")

## 3. Prepare Training Data (Positive and Negative Pairs)

In [ ]:
def create_positive_pairs(data, min_text_len=20, max_text_len=500):
    """
    Create positive training pairs where context → response is causally consistent.
    Uses the natural context-response structure from CRD3.
    """
    positive_pairs = []
    
    print("Creating positive pairs (causal consistency)...")
    for entry in tqdm(data, desc="Processing entries"):
        context = entry['context'].strip()
        response = entry['text'].strip()
        
        # Quality filters
        if len(context) < min_text_len or len(response) < min_text_len:
            continue
        if len(context) > max_text_len or len(response) > max_text_len:
            continue
        
        # Avoid very similar context and response (likely duplicates)
        if context.lower() == response.lower():
            continue
        
        positive_pairs.append({
            'premise': context,
            'hypothesis': response,
            'label': 2,  # Entailment (NLI label)
            'episode': entry['episode'],
            'character': entry['character']
        })
    
    print(f"\n✓ Created {len(positive_pairs):,} positive pairs")
    return positive_pairs


def create_negative_pairs(positive_pairs, ratio=1.0):
    """
    Create negative pairs by mismatching contexts with unrelated responses.
    Ensures different episodes for strong negative examples.
    """
    n_negatives = int(len(positive_pairs) * ratio)
    negative_pairs = []
    
    print(f"\nCreating {n_negatives:,} negative pairs (non-causal)...")
    
    # Group by episode for better negative sampling
    episodes = {}
    for pair in positive_pairs:
        ep = pair['episode']
        if ep not in episodes:
            episodes[ep] = []
        episodes[ep].append(pair)
    
    episode_list = list(episodes.keys())
    
    for i in tqdm(range(n_negatives), desc="Generating negatives"):
        # Select two different episodes for strong negatives
        ep1, ep2 = random.sample(episode_list, 2)
        
        pair1 = random.choice(episodes[ep1])
        pair2 = random.choice(episodes[ep2])
        
        # Mismatch premise from one with hypothesis from another
        negative_pairs.append({
            'premise': pair1['premise'],
            'hypothesis': pair2['hypothesis'],
            'label': 0,  # Contradiction (NLI label)
            'episode': f"{ep1}+{ep2}",
            'character': f"{pair1['character']}+{pair2['character']}"
        })
    
    print(f"✓ Created {len(negative_pairs):,} negative pairs")
    return negative_pairs


# Create training pairs
positive_pairs = create_positive_pairs(crd3_data)
negative_pairs = create_negative_pairs(positive_pairs, ratio=1.0)

In [ ]:
# Combine and shuffle all pairs
all_pairs = positive_pairs + negative_pairs
random.shuffle(all_pairs)

print("="*60)
print("TRAINING DATA SUMMARY")
print("="*60)
print(f"\nTotal pairs: {len(all_pairs):,}")
print(f"  Positive (entailment): {len(positive_pairs):,} ({len(positive_pairs)/len(all_pairs)*100:.1f}%)")
print(f"  Negative (contradiction): {len(negative_pairs):,} ({len(negative_pairs)/len(all_pairs)*100:.1f}%)")

# Display sample pairs
print("\n" + "="*60)
print("SAMPLE POSITIVE PAIR (Causal Consistency)")
print("="*60)
pos_sample = positive_pairs[0]
print(f"Context: {pos_sample['premise'][:200]}...")
print(f"Response: {pos_sample['hypothesis'][:200]}...")
print(f"Label: {pos_sample['label']} (Entailment)")

print("\n" + "="*60)
print("SAMPLE NEGATIVE PAIR (No Causal Link)")
print("="*60)
neg_sample = negative_pairs[0]
print(f"Context: {neg_sample['premise'][:200]}...")
print(f"Response: {neg_sample['hypothesis'][:200]}...")
print(f"Label: {neg_sample['label']} (Contradiction)")

In [ ]:
# Create train/validation/test splits
train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1

n_total = len(all_pairs)
n_train = int(n_total * train_ratio)
n_val = int(n_total * val_ratio)

train_data = all_pairs[:n_train]
val_data = all_pairs[n_train:n_train + n_val]
test_data = all_pairs[n_train + n_val:]

print("\nDataset splits:")
print(f"  Train: {len(train_data):,} pairs ({len(train_data)/n_total*100:.1f}%)")
print(f"  Validation: {len(val_data):,} pairs ({len(val_data)/n_total*100:.1f}%)")
print(f"  Test: {len(test_data):,} pairs ({len(test_data)/n_total*100:.1f}%)")

# Convert to HuggingFace datasets
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)
test_dataset = Dataset.from_list(test_data)

dataset_dict = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
    'test': test_dataset
})

print("\n✓ Datasets created successfully")
print(dataset_dict)

## 4. Model Setup and Tokenization

In [ ]:
# Initialize DeBERTa-v3-small model (lightweight version)
MODEL_NAME = "microsoft/deberta-v3-small"

print(f"Loading model: {MODEL_NAME}")
print("This is a lightweight model suitable for Kaggle training.\n")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,  # 3-way classification: contradiction, neutral, entailment
    problem_type="single_label_classification"
)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Total parameters: {model.num_parameters():,}")
print(f"  Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# Tokenization function
def tokenize_function(examples):
    """
    Tokenize premise-hypothesis pairs for NLI.
    DeBERTa format: [CLS] premise [SEP] hypothesis [SEP]
    """
    return tokenizer(
        examples['premise'],
        examples['hypothesis'],
        truncation=True,
        max_length=256,
        padding=False  # Padding will be done by data collator
    )

# Tokenize datasets
print("Tokenizing datasets...")
tokenized_datasets = dataset_dict.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing",
    remove_columns=['premise', 'hypothesis', 'episode', 'character']
)

# Rename label column for Trainer compatibility
tokenized_datasets = tokenized_datasets.rename_column('label', 'labels')

print("\n✓ Tokenization complete")
print(tokenized_datasets)

## 5. Training Configuration and Metrics

In [ ]:
# Define evaluation metrics
def compute_metrics(eval_pred):
    """
    Compute comprehensive metrics for model evaluation.
    """
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=-1)
    
    # Overall accuracy
    accuracy = accuracy_score(labels, preds)
    
    # Precision, Recall, F1 for each class
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='weighted', zero_division=0
    )
    
    # Per-class metrics
    precision_per_class, recall_per_class, f1_per_class, _ = precision_recall_fscore_support(
        labels, preds, average=None, zero_division=0
    )
    
    metrics = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'precision_contradiction': precision_per_class[0] if len(precision_per_class) > 0 else 0,
        'precision_neutral': precision_per_class[1] if len(precision_per_class) > 1 else 0,
        'precision_entailment': precision_per_class[2] if len(precision_per_class) > 2 else 0,
        'recall_contradiction': recall_per_class[0] if len(recall_per_class) > 0 else 0,
        'recall_neutral': recall_per_class[1] if len(recall_per_class) > 1 else 0,
        'recall_entailment': recall_per_class[2] if len(recall_per_class) > 2 else 0,
        'f1_contradiction': f1_per_class[0] if len(f1_per_class) > 0 else 0,
        'f1_neutral': f1_per_class[1] if len(f1_per_class) > 1 else 0,
        'f1_entailment': f1_per_class[2] if len(f1_per_class) > 2 else 0,
    }
    
    return metrics

print("✓ Metrics function defined")

In [ ]:
# Training arguments
OUTPUT_DIR = "./causal_critic_deberta_v3_small"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    
    # Training hyperparameters
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    
    # Evaluation and saving
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    
    # Logging
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=100,
    report_to="none",  # Disable wandb/tensorboard for Kaggle
    
    # Performance optimization
    fp16=torch.cuda.is_available(),  # Mixed precision if GPU available
    dataloader_num_workers=2,
    gradient_accumulation_steps=2,
    
    # Reproducibility
    seed=42,
)

print("Training Configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  FP16: {training_args.fp16}")
print(f"  Output directory: {OUTPUT_DIR}")

## 6. Train the Model

In [ ]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("✓ Trainer initialized")
print("\nStarting training...")
print("="*60)

In [ ]:
# Train the model
train_result = trainer.train()

print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"\nTraining time: {train_result.metrics['train_runtime']:.2f} seconds")
print(f"Training samples/second: {train_result.metrics['train_samples_per_second']:.2f}")
print(f"Final training loss: {train_result.metrics['train_loss']:.4f}")

In [ ]:
# Save the final model
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"\n✓ Model and tokenizer saved to: {OUTPUT_DIR}")

## 7. Comprehensive Model Evaluation

In [ ]:
# Evaluate on validation set
print("Evaluating on validation set...")
val_results = trainer.evaluate(tokenized_datasets['validation'])

print("\n" + "="*60)
print("VALIDATION SET RESULTS")
print("="*60)
for key, value in val_results.items():
    if key.startswith('eval_'):
        print(f"{key[5:]}: {value:.4f}")

In [ ]:
# Evaluate on test set
print("\nEvaluating on test set...")
test_results = trainer.evaluate(tokenized_datasets['test'])

print("\n" + "="*60)
print("TEST SET RESULTS")
print("="*60)
for key, value in test_results.items():
    if key.startswith('eval_'):
        print(f"{key[5:]}: {value:.4f}")

In [ ]:
# Get predictions for detailed analysis
print("\nGenerating predictions for detailed analysis...")
predictions_output = trainer.predict(tokenized_datasets['test'])
predictions = np.argmax(predictions_output.predictions, axis=-1)
true_labels = predictions_output.label_ids

# Determine which labels are actually present in the data
unique_labels = sorted(np.unique(np.concatenate([true_labels, predictions])))
all_label_names = ['Contradiction', 'Neutral', 'Entailment']
present_label_names = [all_label_names[i] for i in unique_labels]

# Confusion matrix with only present labels
cm = confusion_matrix(true_labels, predictions, labels=unique_labels)

print("\n" + "="*60)
print("CONFUSION MATRIX")
print("="*60)
print(f"Present labels: {present_label_names}")
print(f"Label mapping: {dict(zip(unique_labels, present_label_names))}")
cm_df = pd.DataFrame(cm, index=present_label_names, columns=present_label_names)
print(cm_df)

# Visualize confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=present_label_names, yticklabels=present_label_names,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - Test Set', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Detailed classification report
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
report = classification_report(true_labels, predictions, 
                               labels=unique_labels,
                               target_names=present_label_names, 
                               digits=4)
print(report)

In [ ]:
# Per-class performance visualization
report_dict = classification_report(true_labels, predictions, 
                                    labels=unique_labels,
                                    target_names=present_label_names, 
                                    output_dict=True)

metrics_df = pd.DataFrame({
    'Precision': [report_dict[label]['precision'] for label in present_label_names],
    'Recall': [report_dict[label]['recall'] for label in present_label_names],
    'F1-Score': [report_dict[label]['f1-score'] for label in present_label_names]
}, index=present_label_names)

# Plot per-class metrics
fig, ax = plt.subplots(figsize=(12, 6))
metrics_df.plot(kind='bar', ax=ax, width=0.8)
ax.set_title('Per-Class Performance Metrics', fontsize=14, fontweight='bold')
ax.set_xlabel('Class', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_ylim(0, 1.1)
ax.legend(loc='lower right')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nPer-Class Metrics:")
print(metrics_df)

## 8. Qualitative Analysis - Sample Predictions

In [ ]:
# Analyze sample predictions
def analyze_predictions(dataset, predictions, true_labels, n_samples=10):
    """
    Display sample predictions with context for qualitative analysis.
    """
    label_map = {0: 'Contradiction', 1: 'Neutral', 2: 'Entailment'}
    
    # Get indices for correct and incorrect predictions
    correct_idx = np.where(predictions == true_labels)[0]
    incorrect_idx = np.where(predictions != true_labels)[0]
    
    print("="*80)
    print(f"CORRECT PREDICTIONS (Random {n_samples//2} samples)")
    print("="*80)
    
    for i in np.random.choice(correct_idx, min(n_samples//2, len(correct_idx)), replace=False):
        print(f"\nSample {i}:")
        print(f"  True Label: {label_map[true_labels[i]]}")
        print(f"  Predicted: {label_map[predictions[i]]}")
        print(f"  ✓ CORRECT")
        print("-" * 80)
    
    print("\n" + "="*80)
    print(f"INCORRECT PREDICTIONS (Random {n_samples//2} samples)")
    print("="*80)
    
    for i in np.random.choice(incorrect_idx, min(n_samples//2, len(incorrect_idx)), replace=False):
        print(f"\nSample {i}:")
        print(f"  True Label: {label_map[true_labels[i]]}")
        print(f"  Predicted: {label_map[predictions[i]]}")
        print(f"  ✗ INCORRECT")
        print("-" * 80)

analyze_predictions(test_dataset, predictions, true_labels, n_samples=10)

## 9. Test with Custom Examples

In [ ]:
# Create a prediction function
def predict_causal_consistency(premise, hypothesis):
    """
    Predict causal consistency between premise (context) and hypothesis (response).
    Returns label and probability distribution.
    """
    inputs = tokenizer(premise, hypothesis, truncation=True, 
                      max_length=256, return_tensors="pt")
    
    # Move to same device as model
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)[0]
        predicted_label = torch.argmax(probs).item()
    
    label_map = {0: 'Contradiction', 1: 'Neutral', 2: 'Entailment'}
    
    return {
        'label': label_map[predicted_label],
        'label_id': predicted_label,
        'probabilities': {
            'Contradiction': probs[0].item(),
            'Neutral': probs[1].item(),
            'Entailment': probs[2].item()
        },
        'entailment_score': probs[2].item()  # Causal consistency score
    }

print("✓ Prediction function ready")

In [ ]:
# Test with D&D scenarios
test_scenarios = [
    {
        'premise': "I cast Fireball at the goblin horde charging towards us.",
        'hypothesis': "The goblins scatter as flames engulf them. Three fall instantly, charred beyond recognition.",
        'expected': 'High causal consistency (Entailment)'
    },
    {
        'premise': "I search the ancient library for information about the Dragon's Curse.",
        'hypothesis': "You find a dusty tome describing the ritual needed to break the curse.",
        'expected': 'High causal consistency (Entailment)'
    },
    {
        'premise': "I attempt to persuade the guard to let us pass.",
        'hypothesis': "A dragon suddenly swoops down and attacks the village.",
        'expected': 'Low causal consistency (Contradiction)'
    },
    {
        'premise': "I heal my wounded companion with a spell.",
        'hypothesis': "The weather turns stormy as dark clouds gather.",
        'expected': 'Low causal consistency (Contradiction/Neutral)'
    },
    {
        'premise': "I roll to investigate the suspicious noise in the corridor.",
        'hypothesis': "You notice a hidden door slightly ajar, with faint light seeping through.",
        'expected': 'High causal consistency (Entailment)'
    }
]

print("="*80)
print("TESTING CAUSAL CRITIC WITH D&D SCENARIOS")
print("="*80)

for i, scenario in enumerate(test_scenarios, 1):
    result = predict_causal_consistency(scenario['premise'], scenario['hypothesis'])
    
    print(f"\nTest {i}: {scenario['expected']}")
    print(f"Context: {scenario['premise'][:100]}...")
    print(f"Response: {scenario['hypothesis'][:100]}...")
    print(f"\nPrediction: {result['label']}")
    print(f"Causal Consistency Score: {result['entailment_score']:.4f}")
    print(f"\nProbabilities:")
    for label, prob in result['probabilities'].items():
        print(f"  {label}: {prob:.4f}")
    print("-" * 80)

## 10. Training History Visualization

In [ ]:
# Extract training history from logs
import os

log_history = trainer.state.log_history

# Separate training and evaluation logs
train_logs = [log for log in log_history if 'loss' in log and 'eval_loss' not in log]
eval_logs = [log for log in log_history if 'eval_loss' in log]

# Extract metrics
train_steps = [log['step'] for log in train_logs if 'step' in log]
train_loss = [log['loss'] for log in train_logs if 'loss' in log]

eval_steps = [log['step'] for log in eval_logs if 'step' in log]
eval_loss = [log['eval_loss'] for log in eval_logs if 'eval_loss' in log]
eval_accuracy = [log['eval_accuracy'] for log in eval_logs if 'eval_accuracy' in log]
eval_f1 = [log['eval_f1'] for log in eval_logs if 'eval_f1' in log]

# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curves
axes[0].plot(train_steps, train_loss, label='Training Loss', linewidth=2)
axes[0].plot(eval_steps, eval_loss, label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Steps', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy curve
axes[1].plot(eval_steps, eval_accuracy, label='Validation Accuracy', 
            color='green', linewidth=2)
axes[1].set_xlabel('Steps', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Validation Accuracy', fontsize=14, fontweight='bold')
axes[1].set_ylim(0, 1.1)
axes[1].legend()
axes[1].grid(alpha=0.3)

# F1 Score curve
axes[2].plot(eval_steps, eval_f1, label='Validation F1', 
            color='orange', linewidth=2)
axes[2].set_xlabel('Steps', fontsize=12)
axes[2].set_ylabel('F1 Score', fontsize=12)
axes[2].set_title('Validation F1 Score', fontsize=14, fontweight='bold')
axes[2].set_ylim(0, 1.1)
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Training history visualized")

## 11. Final Summary and Model Export

In [ ]:
# Create comprehensive summary report
summary_report = {
    'model_name': MODEL_NAME,
    'task': 'Causal Consistency Evaluation (NLI)',
    'dataset': {
        'source': 'CRD3 NPC Dialogues',
        'total_pairs': len(all_pairs),
        'positive_pairs': len(positive_pairs),
        'negative_pairs': len(negative_pairs),
        'train_size': len(train_data),
        'val_size': len(val_data),
        'test_size': len(test_data)
    },
    'training': {
        'epochs': training_args.num_train_epochs,
        'batch_size': training_args.per_device_train_batch_size,
        'learning_rate': training_args.learning_rate,
        'total_training_time': train_result.metrics['train_runtime']
    },
    'performance': {
        'test_accuracy': test_results['eval_accuracy'],
        'test_f1': test_results['eval_f1'],
        'test_precision': test_results['eval_precision'],
        'test_recall': test_results['eval_recall'],
        'per_class': {
            'contradiction': {
                'precision': test_results['eval_precision_contradiction'],
                'recall': test_results['eval_recall_contradiction'],
                'f1': test_results['eval_f1_contradiction']
            },
            'neutral': {
                'precision': test_results['eval_precision_neutral'],
                'recall': test_results['eval_recall_neutral'],
                'f1': test_results['eval_f1_neutral']
            },
            'entailment': {
                'precision': test_results['eval_precision_entailment'],
                'recall': test_results['eval_recall_entailment'],
                'f1': test_results['eval_f1_entailment']
            }
        }
    }
}

# Save summary report
with open(f'{OUTPUT_DIR}/training_summary.json', 'w') as f:
    json.dump(summary_report, f, indent=2)

print("="*80)
print("FINAL TRAINING SUMMARY")
print("="*80)
print(json.dumps(summary_report, indent=2))

print(f"\n✓ Summary report saved to: {OUTPUT_DIR}/training_summary.json")

In [ ]:
# Save configuration for deployment
deployment_config = {
    'model_path': OUTPUT_DIR,
    'model_name': MODEL_NAME,
    'task': 'causal_consistency',
    'method': 'fine-tuned_nli',
    'label_mapping': {
        0: 'contradiction',
        1: 'neutral',
        2: 'entailment'
    },
    'score_interpretation': 'Higher entailment probability = better causal consistency',
    'usage': {
        'input_format': 'premise (context) + hypothesis (response)',
        'output': 'entailment probability (0-1 range)',
        'threshold_high': 0.7,
        'threshold_low': 0.3
    }
}

with open(f'{OUTPUT_DIR}/deployment_config.json', 'w') as f:
    json.dump(deployment_config, f, indent=2)

print(f"✓ Deployment config saved to: {OUTPUT_DIR}/deployment_config.json")

## 12. Usage Instructions

In [ ]:
print("="*80)
print("MODEL USAGE INSTRUCTIONS")
print("="*80)
print("""
The trained Causal Critic model can be used to evaluate causal consistency
between player actions and DM responses in D&D narratives.

Example Usage:
-------------

from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

# Load model and tokenizer
model = AutoModelForSequenceClassification.from_pretrained('./causal_critic_deberta_v3_small')
tokenizer = AutoTokenizer.from_pretrained('./causal_critic_deberta_v3_small')

# Prepare input
context = "I cast Fireball at the goblin horde"
response = "The goblins scatter as flames engulf them"

inputs = tokenizer(context, response, truncation=True, max_length=256, return_tensors="pt")

# Get prediction
with torch.no_grad():
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)[0]
    causal_score = probs[2].item()  # Entailment probability

print(f"Causal Consistency Score: {causal_score:.4f}")

Interpretation:
--------------
- Score > 0.7: High causal consistency (response follows logically from context)
- Score 0.3-0.7: Moderate consistency (neutral or partially related)
- Score < 0.3: Low consistency (response doesn't follow from context)

Model Files:
-----------
- pytorch_model.bin: Trained model weights
- config.json: Model configuration
- tokenizer files: Tokenizer configuration and vocab
- training_summary.json: Performance metrics
- deployment_config.json: Usage instructions
""")
print("="*80)

In [ ]:
print("\n" + "="*80)
print("✓ TRAINING COMPLETE")
print("="*80)
print(f"\nModel saved to: {OUTPUT_DIR}")
print(f"Test Accuracy: {test_results['eval_accuracy']:.4f}")
print(f"Test F1 Score: {test_results['eval_f1']:.4f}")
print(f"\nThe Causal Critic is ready for deployment!")